# 📌 Traccia: Classificazione – Random Forest vs StackingClassifier (Logistic Regression + SVC)
- Tipo di problema: Classificazione binaria
- Dataset: Dataset reale wine da sklearn.datasets, convertito a classificazione binaria: Usare solo le classi 0 e 1 (target < 2) ~130 campioni, 13 feature numeriche

1. Pipeline 1 (modello singolo):
- Preprocessing: StandardScaler
- Riduzione dimensionalità: PCA (n_components=5)
- Modello: RandomForestClassifier (ottimizzazione di n_estimators e max_depth)

2. Pipeline 2 (stacking):
- Preprocessing: MinMaxScaler
- Riduzione dimensionalità: SelectKBest con f_classif (k=5)
- Modello: StackingClassifier con:
- Base estimators: LogisticRegression, SVC
- Final estimator: LogisticRegression
- Ottimizzazione degli iperparametri dei base models (C) e del metaclassificatore

- Metrica di valutazione:
Accuracy (classi bilanciate)

- Valutazione tramite Nested Cross-Validation:
- Outer CV: 5-fold
- Inner CV: 3-fold per tuning iperparametri in ciascuna pipeline

✅ Obiettivo dello studente:
- Caricare il dataset e convertirlo in binario
- Implementare entrambe le pipeline: una con modello singolo, l’altra con stacking
- Applicare nested CV e confrontare le performance usando accuracy
- Discutere se il modello stacked migliora la predizione e in quali condizioni conviene usarlo

📎 Nota didattica:
Questa traccia permette di:
- Introdurre StackingClassifier come tecnica avanzata di ensemble
- Confrontare un ensemble implicito (RandomForest) con uno esplicito (stacking di modelli eterogenei)
- Mostrare come combinare modelli di natura diversa e gestirne la complessità

Stimolare ragionamenti su bias-variance tradeoff e complementarietà tra modelli

jupyter nbconvert --to pdf esame_22_luglio.ipynb
http://localhost:8888/notebooks/Desktop/Universita/Magistrale/Primo_anno/Secondo_semestre/Apprendimento_Automatico/Esercitazioni/esame_22_luglio.ipynb


In [1]:
import import_ipynb
from utilities.functions import nested_cv, best_manifold, plot_embedding, train_final_model_from_nested_cv

# Dataset

In [2]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y = True)

print("X shape: ", X.shape)
print("y shape: ", y.shape)

# Escludo le classi 2 e superiori per ottenere un dataset binario
X = X[y < 2]
y = y[y < 2]

print("X shape after filtering: ", X.shape)
print("y shape after filtering: ", y.shape)

X shape:  (178, 13)
y shape:  (178,)
X shape after filtering:  (130, 13)
y shape after filtering:  (130,)


## Dataset splitting

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Pipeline 1
- Preprocessing: StandardScaler
- Riduzione dimensionalità: PCA (n_components=5)
- Modello: RandomForestClassifier (ottimizzazione di n_estimators e max_depth)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('rf', RandomForestClassifier())
])

rf_params_grid = {
    'pca__n_components': [2, 5, 10],
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 5, 10]
}

In [5]:
rf_nested_cv_result = nested_cv(
    model = rf_pipeline,
    param_grid = rf_params_grid,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['accuracy']
)


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'pca__n_components': 10, 'rf__max_depth': None, 'rf__n_estimators': 100}
  Calculating metrics on the outer test set...
    ACCURACY: 0.9048

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'pca__n_components': 10, 'rf__max_depth': None, 'rf__n_estimators': 200}
  Calculating metrics on the outer test set...
    ACCURACY: 0.9524

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'pca__n_components': 10, 'rf__max_depth': 5, 'rf__n_estimators': 50}
  Calculating metrics on the outer test set...
    ACCURACY: 0.9524

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'pca__n_components': 10, 'rf__max_depth': None, 'rf__n_estimators': 200}
  Calculating metrics on the outer test set...
    ACCURACY:

### Test del modello finale di Pipeline 1 con migliori iperaparametri stimati

In [6]:
final_model, final_params, test_metrics = train_final_model_from_nested_cv(
    model = rf_pipeline,
    all_fold_best_params = rf_nested_cv_result["all_fold_best_params"],
    X = X_train,
    y = y_train,
    X_test = X_test,
    y_test = y_test,
    score_per_fold = rf_nested_cv_result["performance_summary"]["accuracy"]["all_scores"],
    strategy = 'most_frequent',
    scoring = ['accuracy']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'pca__n_components': 10, 'rf__max_depth': None, 'rf__n_estimators': 200}

Calcolo delle metriche sul test set finale...
  ACCURACY: 0.9615


## Pipeline 2
- Preprocessing: MinMaxScaler
- Riduzione dimensionalità: SelectKBest con f_classif (k=5)
- Modello: StackingClassifier con:
- Base estimators: LogisticRegression, SVC
- Final estimator: LogisticRegression
- Ottimizzazione degli iperparametri dei base models (C) e del metaclassificatore

In [7]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

stacking_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('stacking', StackingClassifier(
        estimators=[
            ('lr', LogisticRegression()),
            ('svc', SVC())
        ],
        final_estimator=LogisticRegression()
    ))
])

stacking_params_grid = {
    'selector__k': [2, 5, 10],
    'stacking__lr__C': [0.01, 0.1, 1, 10],
    'stacking__svc__C': [0.01, 0.1, 1, 10],
    'stacking__final_estimator__C': [0.01, 0.1, 1, 10]
}

In [8]:
stacking_nested_cv_result = nested_cv(
    model = stacking_pipeline,
    param_grid = stacking_params_grid,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['accuracy']
)


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'selector__k': 10, 'stacking__final_estimator__C': 1, 'stacking__lr__C': 0.01, 'stacking__svc__C': 0.1}
  Calculating metrics on the outer test set...
    ACCURACY: 1.0000

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'selector__k': 5, 'stacking__final_estimator__C': 0.1, 'stacking__lr__C': 0.01, 'stacking__svc__C': 10}
  Calculating metrics on the outer test set...
    ACCURACY: 0.9048

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'selector__k': 10, 'stacking__final_estimator__C': 0.1, 'stacking__lr__C': 0.01, 'stacking__svc__C': 10}
  Calculating metrics on the outer test set...
    ACCURACY: 1.0000

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'selector__k': 10, 'stacking__final

### Test del modello finale di Pipeline 2 con migliori iperaparametri stimati

In [10]:
final_model, final_params, test_metrics = train_final_model_from_nested_cv(
    model = stacking_pipeline,
    all_fold_best_params = stacking_nested_cv_result["all_fold_best_params"],
    X = X_train,
    y = y_train,
    X_test = X_test,
    y_test = y_test,
    score_per_fold = stacking_nested_cv_result["performance_summary"]["accuracy"]["all_scores"],
    strategy = 'most_frequent',
    scoring = ['accuracy']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'stacking__final_estimator__C': 1, 'stacking__svc__C': 0.1, 'selector__k': 10, 'stacking__lr__C': 0.01}

Calcolo delle metriche sul test set finale...
  ACCURACY: 1.0000
